In [331]:
import pandas as pd

genes = pd.read_csv('data/genes.csv')
pl = pd.read_csv('data/PL.csv')
nl = pd.read_csv('data/NL.csv')
uk = pd.read_csv('data/UK.csv')
us = pd.read_csv('data/US.csv')

What can go wrong?

- wrong class of an item
- some values are missing
- missing values are coded differently (eg. NA, N/A, missing, '' etc.) 
- id values are duplicated
- unit inconsistencies in weight and/or height
- missplaced dots
- weight or height are outside of the possible range of values
- there is more than 2 distinct kinds of sex or smoking status
- swapped values between columns

As NL dataset is correct, I will start with inspecting its structure for reference.

## NL dataset

In [262]:
nl['sex'] = pd.Categorical(nl['sex'])
nl['smokes'] = pd.Categorical(nl['smokes'])
nl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1322 entries, 0 to 1321
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   id      1322 non-null   object  
 1   weight  1322 non-null   int64   
 2   height  1322 non-null   int64   
 3   sex     1322 non-null   category
 4   smokes  1322 non-null   category
dtypes: category(2), int64(2), object(1)
memory usage: 33.9+ KB


## UK dataset

In [263]:
uk.info()
uk.apply(lambda x: x.unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694 entries, 0 to 693
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      694 non-null    object 
 1   weight  694 non-null    float64
 2   height  694 non-null    float64
 3   sex     694 non-null    object 
 4   smokes  694 non-null    int64  
dtypes: float64(2), int64(1), object(2)
memory usage: 27.2+ KB


id        [UK_0001, UK_0002, UK_0003, UK_0004, UK_0005, ...
weight    [180.8, 141.1, 218.3, 147.7, 216.0, 103.6, 200...
height    [-1.0, 65.7, 68.5, 68.9, 62.6, 63.0, 61.4, 67....
sex       [male, female, FEMALE, Woman, Male, MALE, Female]
smokes                                               [0, 1]
dtype: object

Identified errors:
- sex naming is inconsistent
- both sex and smokes would be better as categories
- different smoking status unit

In [264]:
uk['sex'] = uk['sex'].str.lower()
uk.loc[uk['sex'] == 'woman', 'sex'] = 'female'
uk['sex'].unique()



array(['male', 'female'], dtype=object)

In [265]:

uk['sex'] = pd.Categorical(uk['sex'])
uk['smokes'] = uk['smokes'].map({1: 'yes', 0: 'no'})
uk['smokes'] = pd.Categorical(uk['smokes'])
uk.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 694 entries, 0 to 693
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   id      694 non-null    object  
 1   weight  694 non-null    float64 
 2   height  694 non-null    float64 
 3   sex     694 non-null    category
 4   smokes  694 non-null    category
dtypes: category(2), float64(2), object(1)
memory usage: 18.0+ KB


In [266]:
any(uk['id'].duplicated())

False

There are no duplicated ID's as well. Let's look at the structure for numeric variables.


In [267]:
print(uk['weight'].describe())
print(uk['height'].describe())

count    694.000000
mean     273.200865
std      264.109384
min       61.700000
25%      158.700000
50%      185.200000
75%      211.600000
max      999.000000
Name: weight, dtype: float64
count    694.000000
mean      64.406484
std       14.566331
min       -1.000000
25%       64.600000
50%       67.300000
75%       69.700000
max       78.000000
Name: height, dtype: float64


Identified issues:

- there is negative height, which is impossible
- height and weight are in british measurement units
- some weights are outside of range of possible values

In [268]:
uk['weight'] = uk['weight']*0.453592

In [269]:

uk['weight'].describe()

#it seems quie impossible for somebody to weight above 200 kg

uk.loc[uk['weight'] > 400, 'weight']

#I'll assume the weight 453.14 kg to be missing value and replace it with mean

uk.loc[uk['weight'] > 400, 'weight'] = uk['weight'].median()

uk['weight'].describe()

count    694.000000
mean      81.370353
std       14.505905
min       27.986626
25%       71.985050
50%       84.005238
75%       89.992653
max      129.001565
Name: weight, dtype: float64

In [270]:
uk['height'] = uk['height']*2.54
uk['height'].describe()

count    694.000000
mean     163.592470
std       36.998482
min       -2.540000
25%      164.084000
50%      170.942000
75%      177.038000
max      198.120000
Name: height, dtype: float64

In [271]:
uk.loc[uk['height'] < 140, 'height'] = uk['height'].median()
uk['height'].describe()

count    694.000000
mean     171.341666
std        8.748447
min      148.082000
25%      166.116000
50%      170.942000
75%      177.038000
max      198.120000
Name: height, dtype: float64

In [272]:
num_cols = uk.select_dtypes(include="number").columns
uk[num_cols] = uk[num_cols].round(3)


## PL dataset

Identified errors:

- missing data are present
- smoking is recorded inconsistently
- heigth is written with an error
- ids are duplicated

To practice a different approach, this time missing values will be removed.

In [273]:
pl.apply(lambda x: x.isna().sum())
pl.dropna(inplace = True)
pl.isna().sum()

id        0
heigth    0
weight    0
sex       0
smokes    0
dtype: int64

In [274]:
pl['smokes'].unique()


array([False, True], dtype=object)

In [275]:
pl.apply(lambda x: x.unique())
pl['smokes'] = pl['smokes'].map({True: 'yes', False: 'no'})
pl['smokes'] = pd.Categorical(pl['smokes'])
pl['sex'] = pd.Categorical(pl['sex'])
pl.apply(lambda x: x.unique())

id        [S0001, S0002, S0003, S0004, S0005, S0006, S00...
heigth    [178.0, 155.0, 173.0, 167.0, 181.0, 183.0, 166...
weight    [77.0, 74.0, 70.0, 95.0, 78.0, 97.0, 57.0, 81....
sex       ['female', 'male']
Categories (2, object): ['f...
smokes    ['no', 'yes']
Categories (2, object): ['no', '...
dtype: object

In [276]:
pl.describe()

#weight and height seem to be fine

pl['height'] = pl['heigth']
pl.drop('heigth', axis = 1, inplace = True)

num_cols = pl.select_dtypes(include="number").columns
pl[num_cols] = pl[num_cols].round(3)

pl.describe()

,weight,height
count,1306.000000,1306.000000
mean,74.281011,168.682236
std,15.426302,8.980572
min,35.000000,146.000000
25%,65.000000,161.000000
50%,71.000000,167.000000
75%,84.000000,176.000000
max,136.000000,190.000000


In [277]:
#pl.drop(pl.index[pl['id'].duplicated()], axis = 0, inplace = True)
pl.drop_duplicates(subset = 'id', inplace = True)
pl['id'].duplicated().sum()

np.int64(0)

In [278]:
pl.info()
pl.apply(lambda x: x.unique())
pl.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 1107 entries, 0 to 1537
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   id      1107 non-null   object  
 1   weight  1107 non-null   float64 
 2   sex     1107 non-null   category
 3   smokes  1107 non-null   category
 4   height  1107 non-null   float64 
dtypes: category(2), float64(2), object(1)
memory usage: 37.0+ KB


,weight,height
count,1107.000000,1107.000000
mean,75.949413,170.063234
std,16.201968,9.090343
min,35.000000,146.000000
25%,65.000000,163.000000
50%,75.000000,169.000000
75%,86.000000,177.000000
max,136.000000,190.000000


## US dataset

Identified issues:

- almost half of height entries are missing

In [279]:
us.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 461 entries, 0 to 460
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      461 non-null    object 
 1   weight  461 non-null    int64  
 2   height  226 non-null    float64
 3   sex     461 non-null    object 
 4   smokes  461 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 18.1+ KB


In [280]:
#to prevent an immense information loss I will not drop the rows with missing values in height column, but I will replace them with median
#in real case scenario, I would try to find out the reason of missing values and if possible, replace them with more accurate values
us['height'] = us['height'].fillna(us['height'].median())
us.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 461 entries, 0 to 460
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   id      461 non-null    object 
 1   weight  461 non-null    int64  
 2   height  461 non-null    float64
 3   sex     461 non-null    object 
 4   smokes  461 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 18.1+ KB


In [282]:
us.apply(lambda x: x.unique())

#it seems the categories are fine

us[['sex', 'smokes']] = us[['sex', 'smokes']].apply(lambda x: pd.Categorical(x))
us.apply(lambda x: x.unique())

us.describe()

#weights and heights seem to be fine, no outliers, so I will not change them

us['id'].duplicated().sum()

#no duplicated ids, so I can be sure that there are no duplicated rows in the dataset



np.int64(0)

## Merging of the datasets

Identified issues:

- there is more rows of antropocentered data than of the gene data
(as no ids were repeated, and all ids for a given country have a correct beginning, I will assume for some ids gene data or antropocentered data were not collected)


In [306]:
genes['id'].duplicated().sum()

np.int64(0)

In [327]:
antrop = pd.concat([nl, uk, pl, us], ignore_index = True)
antrop.info()

combined_dataset = pd.merge(genes, antrop, on = 'id', how = 'inner')
combined_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3584 entries, 0 to 3583
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   id      3584 non-null   object  
 1   weight  3584 non-null   float64 
 2   height  3584 non-null   float64 
 3   sex     3584 non-null   category
 4   smokes  3584 non-null   category
dtypes: category(2), float64(2), object(1)
memory usage: 91.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3260 entries, 0 to 3259
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype   
---  ------  --------------  -----   
 0   id      3260 non-null   object  
 1   source  3260 non-null   object  
 2   geneA   3260 non-null   float64 
 3   geneB   3260 non-null   float64 
 4   geneC   3260 non-null   float64 
 5   weight  3260 non-null   float64 
 6   height  3260 non-null   float64 
 7   sex     3260 non-null   category
 8   smokes  3260 non-null   category
dtypes: category(2), float64(5), objec

In [344]:
uk['id'].str.startswith('UK').sum() == len(uk['id'])
pl['id'].str.startswith('S0').sum() == len(pl['id'])
us['id'].str.startswith('S1').sum() == len(us['id'])
#nl['id'].str.startswith('NL').sum() == len(nl['id'])



np.True_

In [1]:
#combined_dataset.to_csv('data/combined_dataset.csv', index = False)